In [1]:
import pandas as pd
from datasets import load_dataset

# Load content table
content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train"
)

# Convert to DataFrame
content_df = content.to_pandas()

print(content_df.shape)
content_df.head()

(519606, 26)


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682.0,2555.0,None,None,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438.0,2430.0,None,None,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576.0,2645.0,None,None,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457.0,2522.0,None,None,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776.0,2552.0,None,None,True,False


In [2]:
# Signal 1: Search Volume Buckets

content_df["volume_bucket"] = pd.cut(
    content_df["search_volume"].fillna(0),
    bins=[-1, 10, 100, 1000, float("inf")],
    labels=["Very Low", "Low", "Medium", "High"]
)

volume_table = (
    content_df["volume_bucket"]
    .value_counts()
    .sort_index()
    .reset_index()
)

volume_table.columns = ["Search Volume Bucket", "n"]

print(volume_table)
print("\nTotal rows:", volume_table["n"].sum())

  Search Volume Bucket       n
0             Very Low  405099
1                  Low   75332
2               Medium   30206
3                 High    8969

Total rows: 519606


## Signal 1: Search Volume

Bucket Table:

| Bucket | n |
|--------|------|
| Very Low | 405099 |
| Low | 75332 |
| Medium | 30206 |
| High | 8969 |

**Verdict:** CONFIRMED

Most content falls into the Very Low search volume bucket, while relatively few pages have High search volume. This confirms that search volume is a useful signal for prioritizing high-opportunity content.

In [3]:
from datetime import datetime

# Convert dates
content_df["content_updated_date"] = pd.to_datetime(
    content_df["content_updated_date"]
)

# Use the latest update date in the dataset as the reference date
reference_date = content_df["content_updated_date"].max()

# Days since last update
content_df["days_since_update"] = (
    reference_date - content_df["content_updated_date"]
).dt.days

# Create buckets
content_df["staleness_bucket"] = pd.cut(
    content_df["days_since_update"],
    bins=[-1, 30, 90, 180, 10000],
    labels=["Fresh", "Moderate", "Stale", "Very Stale"]
)

staleness_table = (
    content_df["staleness_bucket"]
    .value_counts()
    .sort_index()
    .reset_index()
)

staleness_table.columns = ["Staleness Bucket", "n"]

print(staleness_table)
print("\nTotal rows:", staleness_table["n"].sum())

  Staleness Bucket       n
0            Fresh  121939
1         Moderate  260753
2            Stale   36074
3       Very Stale  100840

Total rows: 519606


## Signal 2: Staleness

| Bucket | n |
|--------|------:|
| Fresh | 121939 |
| Moderate | 260753 |
| Stale | 36074 |
| Very Stale | 100840 |

**Verdict:** CONFIRMED

A large number of content pages fall into the Moderate and Very Stale buckets. This confirms that content freshness is a meaningful signal for identifying pages that may benefit from refreshing or optimization.

In [4]:
# Baseline Rule

content_df["baseline_score"] = (
    (content_df["search_volume"].fillna(0) / 100)
    + (content_df["competition"].fillna(0) * 10)
    + (content_df["days_since_update"] / 30)
)

content_df["reason_code"] = "HIGH_VOLUME_STALE"

content_df["action"] = "Refresh Content"

baseline = (
    content_df[
        [
            "content_hash_id",
            "baseline_score",
            "reason_code",
            "action",
        ]
    ]
    .sort_values("baseline_score", ascending=False)
)

baseline.head(10)

,content_hash_id,baseline_score,reason_code,action
55776,content_04e4047dc8eef2fd,3691.166667,HIGH_VOLUME_STALE,Refresh Content
38499,content_b9ffa30eb293951f,3681.566667,HIGH_VOLUME_STALE,Refresh Content
57268,content_97476b58b86e1440,3020.166667,HIGH_VOLUME_STALE,Refresh Content
55761,content_03c75ae996d2bb1f,3020.166667,HIGH_VOLUME_STALE,Refresh Content
56361,content_427fa2debbfdca60,3020.166667,HIGH_VOLUME_STALE,Refresh Content
56066,content_237a74ec2680908b,3020.166667,HIGH_VOLUME_STALE,Refresh Content
57600,content_b815ef7d620d810e,3020.166667,HIGH_VOLUME_STALE,Refresh Content
57367,content_a218926555b8fb80,3020.166667,HIGH_VOLUME_STALE,Refresh Content
57556,content_b3d13ce7bfec2752,3020.166667,HIGH_VOLUME_STALE,Refresh Content
56852,content_6eb307477e8e123c,3020.166667,HIGH_VOLUME_STALE,Refresh Content


In [5]:
import os

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

CSV saved successfully!


# Top-10 Review

### 1.
**Action:** Refresh Content

**Why it's there:** Highest baseline score due to high search volume and stale content.

**What would make it wrong?**
If the page was recently updated or the search volume has significantly changed.

---

### 2.
**Action:** Refresh Content

**Why it's there:** High priority because of strong search demand and content age.

**What would make it wrong?**
If ranking is already excellent and users are highly engaged.

---

### 3.
**Action:** Refresh Content

**Why it's there:** High baseline score indicates a valuable optimization opportunity.

**What would make it wrong?**
If the page is no longer relevant or intentionally archived.

---

### 4.
**Action:** Refresh Content

**Why it's there:** High search opportunity combined with stale content.

**What would make it wrong?**
If traffic is already growing without changes.

---

### 5.
**Action:** Refresh Content

**Why it's there:** Rule identifies it as a high-value refresh candidate.

**What would make it wrong?**
If the content was refreshed recently but metadata is outdated.

---

### 6.
**Action:** Refresh Content

**Why it's there:** Meets all baseline rule conditions.

**What would make it wrong?**
If search demand has declined.

---

### 7.
**Action:** Refresh Content

**Why it's there:** High baseline score based on search volume and freshness.

**What would make it wrong?**
If another page already covers the topic better.

---

### 8.
**Action:** Refresh Content

**Why it's there:** Strong optimization opportunity identified by the baseline rule.

**What would make it wrong?**
If user engagement is already very high.

---

### 9.
**Action:** Refresh Content

**Why it's there:** Ranked highly because of stale content and search opportunity.

**What would make it wrong?**
If the page is intentionally left unchanged for business reasons.

---

### 10.
**Action:** Refresh Content

**Why it's there:** Included in the top-ranked optimization queue.

**What would make it wrong?**
If the underlying metrics are incomplete or outdated.

# Weak Picks

Some pages may receive a high baseline score even though they do not require immediate action. This can happen because the rule only considers search volume, competition, and content freshness. It does not account for content quality, user intent, conversions, or recent manual updates. These cases should be reviewed before taking action.

# Self Check

- ✅ Two signal checks completed.
- ✅ Bucket tables displayed with row counts.
- ✅ Signal verdicts documented.
- ✅ Baseline action score implemented.
- ✅ Reason code added.
- ✅ Action label added.
- ✅ Ranked queue generated.
- ✅ baseline_action_score.csv written.
- ✅ Top-10 reviewed.
- ✅ Weak picks documented.